# Lensless Imaging Demo

demo of lensless image reconstrution. we clone the repo, install dependencies, download a trained checkpoint and a demo dataset (can be replaced), reconstruct the maesurements with parameter-free ADMM-100 and the trained modular LeADMM-5, then visualize the results and compute metrics

### Cloning repo

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/arslan-yam/lensless_imaging
    %cd lensless_imaging

### Installing dependencies and imports

In [ ]:
!pip install -q -r requirements.txt

import gdown
import matplotlib.pyplot as plt
from pathlib import Path
from src.datasets.custom_dir import IMAGE_EXTS, read_image
from lensless_helpers.preprocessor import get_roi

## Download weights and dataset

file_modular_prepost_weights is the trained model_best.pth, and dataset is a zipped demo set. the archive must unzip to data/custom_test/ with lensless/, masks/, and lensed/ subfolders

In [ ]:
os.makedirs("saved/modular_prepost", exist_ok=True)
file_modular_prepost_weights = "18i4-3Sb5zqrdKVhE72NMyvIhOgSxwsY2"
best_model_path = "saved/modular_prepost/model_best.pth"
gdown.download(id=file_modular_prepost_weights, output=best_model_path, quiet=False)

dataset = "1nVQ_Lbdz2XCMdzM5l8s-RY_Ui3hWTfs4" #replace with your link
gdown.download(id=dataset, output="custom_test.zip", quiet=False)
!mkdir -p data
!unzip -q -o custom_test.zip -d data/
!ls data/custom_test

## Reconstruction

reconstruct the demo maesurements with two methods. each run writes the reconstructed png files to data/saved/<save_path>/test/.

parameter-free ADMM-100 baseline

In [ ]:
!python inference.py model=admm100 inferencer.from_pretrained=null datasets.test.path=data/custom_test inferencer.save_path=admm100

trained modular LeADMM-5 (pre + post)

In [ ]:
!python inference.py model=modular_prepost inferencer.from_pretrained=saved/modular_prepost/model_best.pth datasets.test.path=data/custom_test inferencer.save_path=modular_prepost

## Visualize reconstructions

for some examples, show the lensless measurement, admm-100, modular leadmm-5, and the ground truth.

In [ ]:
data_dir = Path("data/custom_test")

def find_image(directory, stem):
    directory = Path(directory)
    for ext in IMAGE_EXTS:
        path = directory / f"{stem}{ext}"
        if path.exists():
            return path
    raise FileNotFoundError(f"No image found for {stem} in {directory}")

def read_required(path):
    image = read_image(path)
    if image is None:
        raise FileNotFoundError(f"OpenCV cannot read image: {path}")
    return image

names = sorted(p.stem for p in (data_dir / "lensless").iterdir() if p.suffix.lower() in IMAGE_EXTS)

def show(name):
    lensless = read_required(find_image(data_dir / "lensless", name))
    lensed = read_required(find_image(data_dir / "lensed", name))
    admm = get_roi(read_required(find_image("data/saved/admm100/test", name)))
    modular = get_roi(read_required(find_image("data/saved/modular_prepost/test", name)))
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, img, title in zip(axes, [lensless, admm, modular, lensed], ["lensless measurement", "admm-100", "modular leadmm-5", "ground truth"]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.show()

for name in names[:3]:
    show(name)

## Compute metrics

calculate_metrics.py compares the saved reconstructions against the ground-truth lensed images.

In [ ]:
print("admm-100:")
!python calculate_metrics.py --reconstructions data/saved/admm100/test --dataset data/custom_test

print("modular leadmm-5 (prepost):")
!python calculate_metrics.py --reconstructions data/saved/modular_prepost/test --dataset data/custom_test